<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [2]:
using System;
using System.Collections.Generic;

public interface ICreditCard 
{
    int CardNumber { get; set; }
    string HolderName { get; set; }
    int ExpiryDate { get; set; }

    void GetInfo();
    void Pay(int payment);
    void CheckBalance();
}

public interface IGoldCreditCard : ICreditCard 
{
    int BonusMiles { get; set; }
}

public interface IPremiumCreditCard : ICreditCard 
{
    string SupportAssistant { get; set; }
}

public interface ICorporateCreditCard : ICreditCard
{
    string Company { get; set; }
}

public class CreditCard : ICreditCard 
{
    public int CardNumber { get; set; }
    public string HolderName { get; set; }
    public int ExpiryDate { get; set; }
    protected int _balance;
    public string Bank { get; set; }
    public string Currency { get; set; }

    public delegate void BalanceChangedHandler(string message);
    public event BalanceChangedHandler BalanceChanged;

    public CreditCard(int cardNumber, string holderName, int expiryDate, int startBalance, string bank, string currency) 
    {
        CardNumber = cardNumber;
        HolderName = holderName;
        ExpiryDate = expiryDate;
        _balance = startBalance;
        Bank = bank;
        Currency = currency;
    }

    protected void OnBalanceChanged(string message) 
    {
        BalanceChanged?.Invoke(message);
    }

    public virtual void GetInfo() 
    {
        Console.WriteLine($"Номер: {CardNumber}; Владелец: {HolderName}; Срок: {ExpiryDate}. Банк: {Bank}, Валюта: {Currency}");
    }

    public virtual void Pay(int payment) 
    {
        if (payment > _balance)
        {
            Console.WriteLine($"Недостаточно средств! Баланс: {_balance}");
            return;
        }
        
        _balance -= payment;
        Console.WriteLine($"Оплата покупки на {payment} рублей. Остаток: {_balance}");
        
        OnBalanceChanged($"Баланс изменился. Текущий остаток: {_balance}");
    }

    public virtual void CheckBalance() 
    {
        Console.WriteLine($"Баланс кредитной карты: {_balance} рублей");
    }

    public void ShowBankInfo() 
    {
        Console.WriteLine($"Карта выпущена банком {Bank} в валюте {Currency}.");
    }
}

public class GoldCreditCard : CreditCard, IGoldCreditCard 
{
    public int BonusMiles { get; set; }    
    public int DiscountPercent { get; set; } 

    public GoldCreditCard(int cardNumber, string holderName, int expiryDate, int startBalance, int bonusMiles, string bank, string currency) 
        : base (cardNumber, holderName, expiryDate, startBalance, bank, currency)
    {
        BonusMiles = bonusMiles;
        DiscountPercent = 5; 
    }

    public override void Pay(int payment) 
    {
        if (payment > _balance)  
        {
            Console.WriteLine("Недостаточно средств!");
            return;
        }
        
        _balance -= payment;
        int earnedMiles = payment >= 10000 ? 2 : 0;
        BonusMiles += earnedMiles;

        Console.WriteLine($"Покупка на {payment} рублей. Начислено миль: +{earnedMiles}. Всего миль: {BonusMiles}");
        OnBalanceChanged($"Баланс изменился. Текущий остаток: {_balance}");
    }

    public void ApplyDiscount(int amount) 
    {
        int discounted = amount - (amount * DiscountPercent / 100);
        Console.WriteLine($"Скидка {DiscountPercent}% применена. Итог: {discounted} рублей.");
    }
}

public class PremiumCreditCard : CreditCard, IPremiumCreditCard
{
    public string SupportAssistant { get; set;}
    public int ConciergeLevel { get; set; }

    public PremiumCreditCard(int cardNumber, string holderName, int expiryDate, int startBalance, string supportAssistant, string bank, string currency) 
        : base (cardNumber, holderName, expiryDate, startBalance, bank, currency)
    {
        SupportAssistant = supportAssistant;
        ConciergeLevel = 5;
    }

    public override void CheckBalance() 
    {
        if (_balance < 0) 
        {
            Console.WriteLine($"Баланс отрицательный! Желаете получить помощь от {SupportAssistant}?");
        }
        else 
        {
            Console.WriteLine($"Баланс кредитной карты: {_balance} рублей");
        }
    }

    public void CallConcierge() 
    {
        Console.WriteLine($"Консьерж-сервис уровня {ConciergeLevel}: {SupportAssistant} свяжется с вами.");
    }
}

public class CorporateCreditCard : CreditCard, ICorporateCreditCard
{
    public string Company { get; set; }
    public string TaxId { get; set; }

    public CorporateCreditCard(int cardNumber, string holderName, int expiryDate, int startBalance, string company, string bank, string currency) 
        : base (cardNumber, holderName, expiryDate, startBalance, bank, currency)
    {
        Company = company;
        TaxId = "0000000000";
    }

    public override void GetInfo()
    {
        Console.WriteLine($"Номер: {CardNumber}; Владелец: {HolderName}; Срок: {ExpiryDate}; Компания: {Company}");
    }

    public void ShowTaxInfo() 
    {
        Console.WriteLine($"ИНН компании {Company}: {TaxId}");
    }
}

public class StateCorporateCreditCard : CorporateCreditCard 
{
    public string IndividualPowers { get; set; }

    public StateCorporateCreditCard(int cardNumber, string holderName, int expiryDate, int startBalance, string company, string individualPowers, string bank, string currency) 
        : base (cardNumber, holderName, expiryDate, startBalance, company, bank, currency)
    {
        IndividualPowers = individualPowers;
    }

    public override void GetInfo()
    {
        base.GetInfo();
        Console.WriteLine($"Отдельные властные полномочия: {IndividualPowers}");
    }
}

public class CreditCardService
{
    private readonly ICreditCard _card;

    public CreditCardService(ICreditCard card)
    {
        _card = card;
    }

    public void UseCard(int paymentAmount)
    {
        _card.GetInfo();
        _card.Pay(paymentAmount);
        _card.CheckBalance();
    }
}

static void OnBalanceChanged(string message)
{
    Console.WriteLine($"[Событие]: {message}");
}

Console.WriteLine("--- GoldCreditCard ---");
GoldCreditCard gold = new GoldCreditCard(666666, "Гоголь-Моголь", 1025, 26700, 8, "Сбербанк", "RUB");
gold.BalanceChanged += OnBalanceChanged; 

List<ICreditCard> cardsList = new List<ICreditCard> { gold };
foreach (var card in cardsList)
{
    CreditCardService service = new CreditCardService(card);
    service.UseCard(15000);
}

gold.ApplyDiscount(10000);
gold.ShowBankInfo();
Console.WriteLine();

Console.WriteLine("--- PremiumCreditCard ---");
PremiumCreditCard premium = new PremiumCreditCard(545454, "Кузя", 0527, -100, "Создателя", "Альфа-Банк", "USD");
premium.BalanceChanged += OnBalanceChanged;
        
CreditCardService service2 = new CreditCardService(premium);
service2.UseCard(5000); 
premium.CallConcierge(); 
Console.WriteLine();

Console.WriteLine("--- StateCorporateCreditCard ---");
StateCorporateCreditCard statecorporate = new StateCorporateCreditCard(777777, "Землекопьев", 0231, 696969, "ООО БольшиеЛуга", "участвуют в подготовке предложений о формировании государственной политики...","ГосБанк", "RUB");
            
Dictionary<int, ICreditCard> cardsDictionary = new Dictionary<int, ICreditCard>();
cardsDictionary.Add(statecorporate.CardNumber, statecorporate);

if (cardsDictionary.TryGetValue(777777, out ICreditCard foundCard))
{
    foundCard.GetInfo();
    foundCard.Pay(666666);
    foundCard.CheckBalance();
}

--- GoldCreditCard ---
Номер: 666666; Владелец: Гоголь-Моголь; Срок: 1025. Банк: Сбербанк, Валюта: RUB
Покупка на 15000 рублей. Начислено миль: +2. Всего миль: 10
[Событие]: Баланс изменился. Текущий остаток: 11700
Баланс кредитной карты: 11700 рублей
Скидка 5% применена. Итог: 9500 рублей.
Карта выпущена банком Сбербанк в валюте RUB.

--- PremiumCreditCard ---
Номер: 545454; Владелец: Кузя; Срок: 527. Банк: Альфа-Банк, Валюта: USD
Недостаточно средств! Баланс: -100
Баланс отрицательный! Желаете получить помощь от Создателя?
Консьерж-сервис уровня 5: Создателя свяжется с вами.

--- StateCorporateCreditCard ---
Номер: 777777; Владелец: Землекопьев; Срок: 231; Компания: ООО БольшиеЛуга
Отдельные властные полномочия: участвуют в подготовке предложений о формировании государственной политики...
Оплата покупки на 666666 рублей. Остаток: 30303
Баланс кредитной карты: 30303 рублей
